This notebook attempts to solve the following 2D pde with IC and BC:

\begin{equation}
\partial_t\rho = D\Delta \rho
\end{equation}

\begin{equation}
\left\{ \begin{aligned}
    \rho(\mathbf{x},t) &= 1 \qquad \mathbf{x}\in(\delta\Omega_1\cup\delta\Omega_2)\\ 
    \partial_n\rho(\mathbf{x},t) &= 0 \qquad \mathbf{x}\in(\delta\Omega_3\cup\delta\Omega_4)\\ 
    \rho(\mathbf{x},0) &= 0 \qquad \forall\mathbf{x}
\end{aligned}\right.
\end{equation}
% bc opschrijven
where we assume $D$ to be constant independent of time and space.

 Our steps towards a weak form give us a form like this:

$$
\begin{aligned}
\int q(\Delta\rho)da &= \int q(\partial_{xx}+\partial_{yy})\rho da \\
&= \int_{\delta\Omega_1\cup\delta\Omega_3}q(\partial_x\rho) dy\bigg|_{x=0}^1+\int_{\delta\Omega_2\cup\delta\Omega_4}q(\partial_y\rho) dx\bigg|_{y=0}^1\\
&\quad -\int (\partial_x\rho)(\partial_xq)da-\int (\partial_y\rho)(\partial_y q)da \\
&= -\int_{\delta\Omega_1}q(\partial_x\rho) dy\bigg|_{x=0}-\int (\nabla\rho)\cdot(\nabla q)da
\end{aligned}
$$

where the boundary condition for the right, top, and bottom parts are substituted to get rid of a lot of the boundary mess, the dirichlet boundary at the left hand side will be enforced with a constraint handler (so we will ignore it from now on in the theory section). We discretize $\rho$ into time steps $n$ and spacial steps $i$:

$$
\rho^n = \sum_i \rho^{i,n}\psi^i
$$


and now if we change test function $q$ with $\psi^j$ we can substitute and use an implicit time scheme to get the full discrete form:

$$
\sum\int \frac{1}{\Delta t}(\rho^{i,n+1}\psi^i-\rho^{i,n}\psi^i)\psi^j da = -D\sum \rho^{i,n+1}\int(\nabla\psi^i)\cdot(\nabla \psi^j)da \\
\implies \quad \sum \left[ \int \psi^i\psi^j da + \Delta t D \int (\nabla \psi^i)\cdot(\nabla\psi^j)da \right]\rho^{i,n+1} = \sum \left[ \int \psi^i\psi^j da\right]\rho^{i,n} \\
$$

which we can rewrite as:

$$
(M+\Delta t L) x^{n+1} = Mx^n \\
Ax^{n+1}=Mx^n
$$

with
$$M_{ij}=\int \psi^i\psi^j dx,\\
L_{ij}=D\int (\nabla \psi^i)\cdot(\nabla\psi^j)dx\\
A_{ij}=M_{ij}+\Delta t L_{ij}$$


In [1]:
using Ferrite, SparseArrays, WriteVTK

In [2]:
# Stop time, time step and Diffusivity
dt = .005;
T = 1;

D = 1

name = "diffusion_2d";

In [ ]:
# generates 100 gridpoints between 
grid = generate_grid(Quadrilateral, (100, 100), Vec((0.0, 0.0)), Vec((1.0, 1.0)));

# generate the same 
ip = Lagrange{RefQuadrilateral, 1}();
qr = QuadratureRule{RefQuadrilateral}(2);
fr = FacetQuadratureRule{RefQuadrilateral};
cellvalues = CellValues(qr, ip);

# for the right and top boundary rest terms of our integration by parts
# this part is not needed since diffusion requires boundary conditions there thus we will fix the rest terms with constraintHandlers

# fqr = FacetQuadratureRule{RefQuadrilateral}(1);
# facetvalues = FacetValues(Float64, fqr, ip);
# right = getfacetset(grid, "right");
# top = getfacetset(grid, "top");

CellValues(scalar, rdim=2, and sdim=2): 4 quadrature points
 Function interpolation: Lagrange{RefQuadrilateral, 2}()
Geometric interpolation: Lagrange{RefQuadrilateral, 1}()^2

In [4]:
# degree of freedom handler, 
# our finished vector is (rho^1, rho^2, ... , rho^N)
dh = DofHandler(grid)
add!(dh, :rho, ip)
close!(dh);

In [5]:
ch = ConstraintHandler(dh)

#create left_slit
addfacetset!(
    grid,
    "left_slit",
    x -> abs(x[1]) < 1e-8 && 0.25 <= x[2] <= 0.75
)

dbc_left = Dirichlet(
    :rho,
    getfacetset(grid, "left_slit"),
    (x, t) -> 1.0
)

add!(ch, dbc_left)
close!(ch)
update!(ch, 0.0)

$$M_{ij}=\int \psi^i\psi^j dx$$

$$L_{ij}=D\int \psi^i(\partial_x\psi^j)dx$$

In [6]:
function doassemble_L!(
    L::SparseMatrixCSC, 
    cellvalues::CellValues, 
    dh::DofHandler
    )

    n_basefuncs = getnbasefunctions(cellvalues)
    Le = zeros(n_basefuncs, n_basefuncs)

    assembler = start_assemble(L)

    for cell in CellIterator(dh)

        fill!(Le, 0)
        reinit!(cellvalues, cell)

        for q_point in 1:getnquadpoints(cellvalues)
            dx = getdetJdV(cellvalues, q_point)

            for i in 1:n_basefuncs
                dpsi_i = shape_gradient(cellvalues, q_point, i)

                for j in 1:n_basefuncs
                    dpsi_j = shape_gradient(cellvalues, q_point, j)
                    #  \psi^i(\partial_x\psi^j)dx
                    Le[i, j] += D * (dpsi_j ⋅ dpsi_i) * dx
                end
            end
        end

        assemble!(assembler, celldofs(cell), Le)
    end

    return L
end


doassemble_L! (generic function with 1 method)

In [7]:
function doassemble_M!(M::SparseMatrixCSC, cellvalues::CellValues, dh::DofHandler)

    # Allocate the element stiffness matrix
    n_basefuncs = getnbasefunctions(cellvalues)
    Me = zeros(n_basefuncs, n_basefuncs)

    # create an assembler
    assembler = start_assemble(M)

    for cell in CellIterator(dh)

        # per cell we take the local assembly matrix Me as zero
        fill!(Me, 0)

        # cellvalues contains tons and now it knows which cell we are looking at
        reinit!(cellvalues, cell)

        for q_point in 1:getnquadpoints(cellvalues)

            # not really dx but quadrature weight thingies
            dx = getdetJdV(cellvalues, q_point)

            for i in 1:n_basefuncs
                psi_i = shape_value(cellvalues, q_point, i)
                for j in 1:n_basefuncs
                    psi_j = shape_value(cellvalues, q_point, j)
                    Me[i, j] += (psi_i * psi_j) * dx
                end
            end
        end

        assemble!(assembler, celldofs(cell), Me)
    end
    return M
end

doassemble_M! (generic function with 1 method)

In [8]:
M = allocate_matrix(dh);
L = allocate_matrix(dh);

M = doassemble_M!(M, cellvalues, dh)
L = doassemble_L!(L, cellvalues, dh)
A = (dt .* L) + M;

In [9]:
rho_n = zeros(ndofs(dh));
apply!(rho_n, ch)

rhsdata = get_rhs_data(ch, A);
apply!(A, ch);


In [10]:
pvd = paraview_collection(joinpath(name, name))

VTKGridFile(joinpath(name, name), dh) do vtk
    write_solution(vtk, dh, rho_n)
    pvd[0.0] = vtk
end

VTKGridFile for the closed file "diffusion_2d\diffusion_2d.vtu".

In [11]:
# finally the timeloop
for (step, t) in enumerate(dt:dt:T)
    #First of all, we need to update the Dirichlet boundary condition values.
    # does not really do anything since the boundary condition is independent of time?
    update!(ch, t)

    #Secondly, we compute the right-hand-side of the problem.
    b = M * rho_n

    #Then, we can apply the boundary conditions of the current time step.
    apply_rhs!(rhsdata, b, ch)

    #Finally, we can solve the time step and save the solution afterwards.
    rho = A \ b
    apply!(rho, ch)   # enforce rho(0,t)=1 exactly ??

    VTKGridFile(joinpath(name, "$(name)_1-$step"), dh) do vtk
        write_solution(vtk, dh, rho)
        pvd[t] = vtk
    end
    #At the end of the time loop, we set the previous solution to the current one and go to the next time step.
    rho_n .= rho
end

In [12]:
vtk_save(pvd);